In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Mundka_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,284.0,NaN,239.0,91.0,83.0,150.0,87.0,121.0,182.0,239.0,417.0,386.0
1,2,382.0,259.0,233.0,153.0,85.0,155.0,85.0,129.0,227.0,195.0,453.0,373.0
2,3,416.0,234.0,190.0,202.0,130.0,186.0,141.0,108.0,236.0,265.0,486.0,370.0
3,4,358.0,NaN,152.0,NaN,112.0,235.0,NaN,131.0,176.0,365.0,421.0,346.0
4,5,345.0,275.0,184.0,176.0,232.0,234.0,109.0,114.0,164.0,NaN,480.0,339.0
5,6,402.0,340.0,192.0,170.0,310.0,157.0,72.0,133.0,141.0,299.0,443.0,331.0
6,7,368.0,293.0,236.0,215.0,254.0,281.0,90.0,134.0,109.0,345.0,408.0,331.0
7,8,389.0,172.0,204.0,NaN,141.0,196.0,70.0,NaN,87.0,253.0,446.0,362.0
8,9,438.0,270.0,128.0,303.0,288.0,211.0,NaN,161.0,42.0,244.0,435.0,356.0
9,10,381.0,177.0,236.0,256.0,266.0,205.0,NaN,161.0,37.0,NaN,310.0,NaN


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    34 non-null     float64
 2   February   31 non-null     float64
 3   March      36 non-null     float64
 4   April      34 non-null     float64
 5   May        37 non-null     float64
 6   June       35 non-null     float64
 7   July       23 non-null     float64
 8   August     25 non-null     float64
 9   September  33 non-null     float64
 10  October    35 non-null     float64
 11  November   33 non-null     float64
 12  December   34 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,284.0,221.483871,239.0,91.000000,83.0,150.0,87.000000,121.0,182.0,239.000000,417.0,386.0
1,2,382.0,259.000000,233.0,153.000000,85.0,155.0,85.000000,129.0,227.0,195.000000,453.0,373.0
2,3,416.0,234.000000,190.0,202.000000,130.0,186.0,73.869565,108.0,236.0,265.000000,486.0,370.0
3,4,358.0,221.483871,152.0,213.323529,112.0,235.0,73.869565,131.0,176.0,365.000000,421.0,346.0
4,5,345.0,275.000000,184.0,176.000000,232.0,234.0,73.869565,114.0,164.0,242.828571,480.0,339.0


In [9]:
df_ml_ready.shape
df_ml_ready.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     float64
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 3.3 KB
